# PLINDER Clustering

For details on the PLINDER columns and naming conventions, see: https://plinder-org.github.io/plinder/examples/2_query_filter_index.html

In [ ]:
MASTER_INTERFACES_DF_PATH = "/projects/ml/datahub/dfs/pdb/2024_12_01/interfaces_df.parquet"
TRAIN_INTERFACES_DF_PATH = "/projects/ml/datahub/dfs/af3_splits/2024_12_16/interfaces_df_train.parquet"

### Setup PLINDER df

In [ ]:
from plinder.core.scores import query_index
import pandas as pd

In [ ]:
# Get the PLINDER dataframe with the columns of interest
cols_of_interest = [
    "system_id",
    "entry_pdb_id",
    "entry_release_date",
    "system_biounit_id",
    "system_protein_chains_asym_id",
    "system_ligand_chains_asym_id",
    "ligand_instance_chain",
    "pli_qcov__50__strong__component",
    "pli_qcov__70__strong__component",
    "pli_qcov__50__weak__component",
    "pli_qcov__70__weak__component",
]
plinder_df = query_index(columns=cols_of_interest)
plinder_df.head()

In [ ]:
# Count unique clusters
print(f"50% strong: {plinder_df['pli_qcov__50__strong__component'].nunique():,}")
print(f"70% strong: {plinder_df['pli_qcov__70__strong__component'].nunique():,}")
print(f"50% weak: {plinder_df['pli_qcov__50__weak__component'].nunique():,}")
print(f"70% weak: {plinder_df['pli_qcov__70__weak__component'].nunique():,}")

In [ ]:
# Split system_protein_chains_asym_id and system_ligand_chains_asym_id into new columns with format [A_1, A_2] etc.

import ast


def convert_chain_list(chain_str):
    """Convert ['1.A', '2.A'] -> ['A_1', 'A_2']"""
    # Safely parse the string to a list
    chains = ast.literal_eval(chain_str) if isinstance(chain_str, str) else chain_str
    out = []
    for item in chains:
        if "." in item:
            num, chain = item.split(".")
            out.append(f"{chain}_{num}")
        else:
            out.append(item)
    return out


plinder_df["protein_chains_iid"] = plinder_df["system_protein_chains_asym_id"].apply(convert_chain_list)
plinder_df["ligand_chains_iid"] = plinder_df["system_ligand_chains_asym_id"].apply(convert_chain_list)

In [ ]:
# View head
plinder_df.head()

In [ ]:
# Create a lookup DataFrame by exploding the protein and ligand units
df_exploded = []
for _, row in plinder_df.iterrows():
    pdb_id = row["entry_pdb_id"]
    assembly_id = row["system_biounit_id"]
    protein_units = row["protein_chains_iid"]
    ligand_units = row["ligand_chains_iid"]

    # Create all valid protein-ligand combinations within the same system
    for protein_unit in protein_units:
        for ligand_unit in ligand_units:
            # Add both orientations: protein->ligand and ligand->protein
            df_exploded.append(
                {
                    "pdb_id": pdb_id,
                    "assembly_id": assembly_id,
                    "pn_unit_1_iid": protein_unit,
                    "pn_unit_2_iid": ligand_unit,
                    "pli_qcov__50__strong__component": row["pli_qcov__50__strong__component"],
                    "pli_qcov__70__strong__component": row["pli_qcov__70__strong__component"],
                    "pli_qcov__50__weak__component": row["pli_qcov__50__weak__component"],
                    "pli_qcov__70__weak__component": row["pli_qcov__70__weak__component"],
                }
            )
            # Add the reverse orientation
            df_exploded.append(
                {
                    "pdb_id": pdb_id,
                    "assembly_id": assembly_id,
                    "pn_unit_1_iid": ligand_unit,
                    "pn_unit_2_iid": protein_unit,
                    "pli_qcov__50__strong__component": row["pli_qcov__50__strong__component"],
                    "pli_qcov__70__strong__component": row["pli_qcov__70__strong__component"],
                    "pli_qcov__50__weak__component": row["pli_qcov__50__weak__component"],
                    "pli_qcov__70__weak__component": row["pli_qcov__70__weak__component"],
                }
            )

lookup_df = pd.DataFrame(df_exploded)
lookup_df.head()

### Load and analyze existing DF

In [ ]:
master_interfaces_df = pd.read_parquet(MASTER_INTERFACES_DF_PATH)

# Remove all columns that begin with "pli_qcov__" (existing PLINDER clusters)
master_interfaces_df = master_interfaces_df.drop(
    columns=[col for col in master_interfaces_df.columns if col.startswith("pli_qcov__")]
)

master_interfaces_df = master_interfaces_df.merge(
    lookup_df, on=["pdb_id", "assembly_id", "pn_unit_1_iid", "pn_unit_2_iid"], how="left"
)

master_interfaces_df.rename(columns={"cluster_label": "cluster_id_plinder"}, inplace=True)
master_interfaces_df.head()

In [ ]:
# Save the interfaces_df back to parquet
# (We comment out to avoid inadvertent overwrites - perform this step with care; it is best-practice to save the master df to a different path)
# print(f"Saving interfaces_df to {MASTER_INTERFACES_DF_PATH}")
# master_interfaces_df.to_parquet(MASTER_INTERFACES_DF_PATH)

In [ ]:
import pandas as pd

INTERFACES_DF_TRAIN_PATH = "/projects/ml/datahub/dfs/af3_splits/2024_12_16/interfaces_df_train.parquet"
KEY_COLUMNS = ["pdb_id", "assembly_id", "pn_unit_1_iid", "pn_unit_2_iid"]
interfaces_df_train = pd.read_parquet(INTERFACES_DF_TRAIN_PATH)

# Remove existing cluster_id_plinder column if it exists in the target df
# (Legacy column from previous PLINDER clustering)
if "cluster_id_plinder" in interfaces_df_train.columns:
    print("Removing existing cluster_id_plinder column from interfaces_train_df")
    interfaces_df_train = interfaces_df_train.drop("cluster_id_plinder", axis=1)

# Remove all columns that begin with "pli_qcov__" (existing PLINDER clusters)
master_interfaces_df = master_interfaces_df.drop(
    columns=[col for col in master_interfaces_df.columns if col.startswith("pli_qcov__")]
)

print(
    f"Loaded {interfaces_df_train.shape[0]} rows and {interfaces_df_train.shape[1]} columns from {INTERFACES_DF_TRAIN_PATH}"
)

# Merge with lookup_df
interfaces_df_train = interfaces_df_train.merge(lookup_df, on=KEY_COLUMNS, how="left")
print(f"Merged with lookup_df. New shape: {interfaces_df_train.shape}")

# Get the PLINDER column names (which are the columns that begin with "pli_qcov__")
plinder_columns = [col for col in interfaces_df_train.columns if col.startswith("pli_qcov__")]
print(f"PLINDER columns: {plinder_columns}")

# Subset to rows where the first PLINDER column is not null
interfaces_df_train = interfaces_df_train[interfaces_df_train[plinder_columns[0]].notna()]
print(f"Subset to rows where the first PLINDER column is not null. New shape: {interfaces_df_train.shape}")

# Drop duplicates on the key columns, logging the number of duplicates removed
num_duplicates = interfaces_df_train.duplicated(subset=KEY_COLUMNS).sum()
print(
    f"Dropping {num_duplicates} duplicate rows (out of {interfaces_df_train.shape[0]} rows, {num_duplicates / interfaces_df_train.shape[0]:.1%})"
)
interfaces_df_train = interfaces_df_train.drop_duplicates(subset=KEY_COLUMNS)

print(f"Final shape: {interfaces_df_train.shape}")

In [ ]:
# For the example_id column, replace "pdb" with "plinder"
interfaces_df_train["example_id"] = interfaces_df_train["example_id"].str.replace("pdb", "plinder")

# Save the new df to the same path, but instead of _train.parquet, _train_plinder.parquet
new_path = INTERFACES_DF_TRAIN_PATH.replace(".parquet", "_plinder.parquet")
print(f"Saving to {new_path}")
interfaces_df_train.to_parquet(new_path)